Nama  : Alfarell Muchamad Yuwanto

NIM   : 240401010037

Kelas : IF401

# Hands On Pertemuan 12 - Market Basket Analysis & Rekomendasi Produk  

##  Generate & Eksplorasi Dataset Transaksi 

In [1]:
import pandas as pd, numpy as np 
import matplotlib.pyplot as plt 
  
np.random.seed(42) 
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega'] 
  
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk 
transaksi = [] 
for _ in range(50): 
    n_item = np.random.randint(2, 6) 
    transaksi.append([str(x) for x in np.random.choice(produk, n_item, replace=False)]) 
  
# Suntikkan pola: Roti sering bersama Selai 
for i in range(0, 20): 
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]: 
        transaksi[i].append('Selai') 
  
print('Contoh transaksi:', transaksi[:3]) 
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [['Keju', 'Roti', 'Mentega', 'Kopi', 'Selai'], ['Roti', 'Kopi', 'Teh', 'Selai', 'Mentega'], ['Kopi', 'Susu', 'Teh']]
Jumlah transaksi: 50


## One-Hot Encoding Transaksi

In [2]:
from mlxtend.preprocessing import TransactionEncoder 
  
te = TransactionEncoder() 
te_ary = te.fit(transaksi).transform(transaksi) 
df = pd.DataFrame(te_ary, columns=[str(c) for c in te.columns_]) 
  
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


##  Cari Frequent Itemset dengan Apriori 

In [3]:
from mlxtend.frequent_patterns import apriori 
  
for ms in [0.05, 0.1, 0.2]: 
    freq = apriori(df, min_support=ms, use_colnames=True) 
    print(f'min_support={ms}: {len(freq)} itemset ditemukan') 
  
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan) 
freq_items = apriori(df, min_support=0.1, use_colnames=True) 
# Pastikan itemset bertipe str Python untuk kompatibilitas mlxtend association_rules
freq_items['itemsets'] = freq_items['itemsets'].apply(lambda x: frozenset(str(i) for i in x))
freq_items = freq_items.sort_values('support', ascending=False) 
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support                 itemsets
5      0.52       frozenset({Selai})
8      0.46         frozenset({Teh})
3      0.42     frozenset({Mentega})
9      0.36       frozenset({Telur})
1      0.34        frozenset({Keju})
0      0.32        frozenset({Gula})
2      0.32        frozenset({Kopi})
4      0.32        frozenset({Roti})
7      0.32        frozenset({Susu})
36     0.24  frozenset({Selai, Teh})


## Bentuk & Saring Aturan Asosiasi

In [4]:
from mlxtend.frequent_patterns import association_rules 
  
rules = association_rules(freq_items, metric='confidence', min_threshold=0.5) 
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False) 
  
print(rules[['antecedents', 'consequents', 
             'support', 'confidence', 'lift']].head(10))

                    antecedents           consequents  support  confidence  \
9        frozenset({Keju, Teh})    frozenset({Telur})     0.12    0.857143   
13  frozenset({Selai, Mentega})     frozenset({Kopi})     0.10    0.625000   
12      frozenset({Roti, Gula})    frozenset({Selai})     0.10    1.000000   
7           frozenset({Sereal})  frozenset({Mentega})     0.14    0.777778   
10      frozenset({Telur, Teh})     frozenset({Keju})     0.12    0.600000   
14     frozenset({Selai, Kopi})  frozenset({Mentega})     0.10    0.714286   
8      frozenset({Keju, Telur})      frozenset({Teh})     0.12    0.750000   
11     frozenset({Selai, Gula})     frozenset({Roti})     0.10    0.500000   
15   frozenset({Mentega, Kopi})    frozenset({Selai})     0.10    0.714286   
1             frozenset({Roti})    frozenset({Selai})     0.22    0.687500   

        lift  
9   2.380952  
13  1.953125  
12  1.923077  
7   1.851852  
10  1.764706  
14  1.700680  
8   1.630435  
11  1.562500  
15  1.

##  Rekomender Sederhana dengan Content-Based Filtering 

In [5]:
from sklearn.metrics.pairwise import cosine_similarity 
  
katalog = pd.DataFrame({ 
    'produk': produk, 
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy', 
                 'Dairy','Minuman','Bumbu','Minuman','Dairy'] 
}) 
  
fitur = pd.get_dummies(katalog['kategori']) 
sim_matrix = cosine_similarity(fitur) 
  
def rekomendasi_serupa(nama_produk, top_n=3): 
    idx = katalog.index[katalog['produk'] == nama_produk][0] 
    skor = list(enumerate(sim_matrix[idx])) 
    skor = sorted(skor, key=lambda x: x[1], reverse=True) 
    skor = [s for s in skor if s[0] != idx][:top_n] 
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist() 
  
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


## Bandingkan Kedua Pendekatan

In [6]:
produk_target = 'Roti' 
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)] 
print('Rekomendasi dari Association Rules:') 
print(rules_terkait[['consequents', 'lift']].head()) 
print('\nRekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
           consequents      lift
12  frozenset({Selai})  1.923077
1   frozenset({Selai})  1.322115

Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


## Kesimpulan

- **Apa yang dipelajari:** Penerapan Market Basket Analysis menggunakan algoritma `Apriori` dan `Association Rules` (`mlxtend`) dengan metrik evaluasi Support, Confidence, dan Lift, serta pembuatan Recommender System sederhana berbasis `Content-Based Filtering` menggunakan `Cosine Similarity`.
- **Temuan utama:** Algoritma Apriori berhasil menangkap pola asosiasi produk yang sering dibeli bersamaan (seperti aturan `{Roti, Gula} -> {Selai}` dengan Confidence 100% dan Lift 1.92). Association Rules berfokus pada hubungan ko-okurensi transaksi lintas kategori (behavior-driven), sedangkan Content-Based Filtering merekomendasikan produk dengan kategori/fitur yang serupa (feature-driven).
- **Keterbatasan / Pertanyaan:** Association Rules memerlukan volume transaksi yang besar dan rentan masalah cold-start untuk produk baru. Content-Based Filtering ramah terhadap produk baru tetapi cenderung menghasilkan rekomendasi yang monoton. Pendekatan Hybrid Recommender System dapat menggabungkan keunggulan kedua metode untuk rekomendasi yang lebih adaptif.